Integrantes:
- Luis Eduardo Grajales
- Valentina Garcia Obando

**INTRODUCCIÓN**

**Análisis de Energía en Señales de EEG para Imaginería Motora ¡Importante!**

Se cuenta con señales de EEG registradas durante tareas de imaginería motora, donde un sujeto debe imaginar el movimiento de su Mano Izquierda (T1) o su Mano Derecha (T2). Se tiene evidencia científica de que la planificación del movimiento genera una desincronización (caída de energía) en las cortezas motoras contralaterales. El objetivo de este informe es identificar mediante análisis estadístico qué canales de EEG permiten diferenciar significativamente entre ambas intenciones motoras.

1. Cálculo de la Amplitud Efectiva (RMS)
Implemente una función que reciba una señal de múltiples canales y épocas (segmentada por clase) y calcule el valor RMS (Root Mean Square).
La función debe primero calcular el RMS para cada época de cada canal de forma individual sobre el eje del tiempo
Posteriormente, la función debe promediar estos valores a través de todas las épocas para obtener un valor representativo del sujeto por cada canal.
2. Construcción de la Base de Datos Poblacional
Calcule la energía (RMS) de cada canal promediada por épocas para una población de 10 sujetos, discriminando entre los dos grupos de estudio: Grupo Mano Izquierda y Grupo Mano Derecha.
Organice la información en dos DataFrames (uno por grupo).
Las columnas deben corresponder a los nombres de los canales (ej. C3, C4, Cz).
Las filas deben corresponder a cada sujeto analizado.
Cada celda contendrá el valor de RMS promedio calculado en el punto anterior.
3. Identificación de Canales Diferenciales mediante Análisis Estadístico
Determine si existe diferencia estadística significativa entre la activación de los canales para cada grupo de tareas (Izquierda vs. Derecha) siguiendo este flujo de validación:
Prueba de Normalidad: Aplique la prueba de Shapiro-Wilk sobre la distribución de los sujetos en cada canal.
Prueba de Homocedasticidad: Realice una prueba de Levene para verificar si las varianzas entre ambos grupos son iguales.
Prueba de Hipótesis:
Si se cumplen los supuestos de normalidad y homocedasticidad, realice una Prueba t de Student para muestras independientes.
De no cumplirse los requisitos, realice un análisis no paramétrico mediante la Prueba U de Mann-Whitney.
Resultados: Identifique y reporte los canales cuyo $p-valor < 0.05$. Estos canales serán considerados como los que entregan información diferencial clave para el control de una interfaz cerebro-computadora (BCI).
Ejemplo de Visualización Esperada
Para el canal con mayor significancia estadística (típicamente C3 o C4), se solicita incluir un diagrama de caja (Boxplot) que compare visualmente la distribución del RMS entre ambos grupos poblacionales.


**DESAROLLO**

In [3]:
pip install mne pandas numpy scipy seaborn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 54.5 MB/s eta 0:00:00


In [10]:
import zipfile

with zipfile.ZipFile("lab3_bioseñales.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/")

In [16]:
import os
import numpy as np
import pandas as pd
import mne
from scipy.stats import shapiro, levene, ttest_ind, mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns

# 1. FUNCIÓN RMS
def calcular_rms(epocas):
    rms_epocas = np.sqrt(np.mean(epocas**2, axis=2))
    return np.mean(rms_epocas, axis=0)


# 2. CONFIGURACIÓN
ruta_base = "/content/lab3_bioseñales"

sujetos = [f"S{i:03d}" for i in range(1, 11)]
runs = [4, 8, 12]

df_T1 = None
df_T2 = None

# 3. PROCESAMIENTO DE DATOS

for sujeto in sujetos:

    print(f"\nProcesando {sujeto}")

    datos_T1 = []
    datos_T2 = []

    for run in runs:

        archivo = os.path.join(
            ruta_base,
            f"{sujeto}R{run:02d}.edf"
        )

        # Verificación (evita errores)
        if not os.path.exists(archivo):
            print(f" No existe: {archivo}")
            continue

        raw = mne.io.read_raw_edf(archivo, preload=True, verbose=False)

        # Filtrado EEG
        raw.filter(8, 30, verbose=False)

        events, event_id = mne.events_from_annotations(raw)

        # Mapear eventos
        event_dict = {}
        for key in event_id:
            if "T1" in key:
                event_dict["T1"] = event_id[key]
            elif "T2" in key:
                event_dict["T2"] = event_id[key]

        if len(event_dict) < 2:
            print(f" Eventos incompletos en {archivo}")
            continue

        epochs = mne.Epochs(
            raw,
            events,
            event_id=event_dict,
            tmin=0,
            tmax=4,
            baseline=None,
            preload=True,
            verbose=False
        )

        datos_T1.append(epochs['T1'].get_data())
        datos_T2.append(epochs['T2'].get_data())

    # si no hay datos
    if len(datos_T1) == 0 or len(datos_T2) == 0:
        print(f" {sujeto} sin datos válidos")
        continue

    # Concatenar runs
    epocas_T1 = np.concatenate(datos_T1, axis=0)
    epocas_T2 = np.concatenate(datos_T2, axis=0)

    # RMS
    rms_T1 = calcular_rms(epocas_T1)
    rms_T2 = calcular_rms(epocas_T2)

    canales = epochs.ch_names

    if df_T1 is None:
        df_T1 = pd.DataFrame(columns=canales)
        df_T2 = pd.DataFrame(columns=canales)

    df_T1.loc[sujeto] = rms_T1
    df_T2.loc[sujeto] = rms_T2


Procesando S001
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]

Procesando S002
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]

Procesando S003
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]

Procesando S004
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]

Procesa

**ANÁLISIS**

In [17]:
# 4. ANÁLISIS ESTADÍSTICO
resultados = []

for canal in df_T1.columns:

    g1 = df_T1[canal].astype(float)
    g2 = df_T2[canal].astype(float)

    # Normalidad
    p1 = shapiro(g1)[1]
    p2 = shapiro(g2)[1]
    normal = (p1 > 0.05) and (p2 > 0.05)

    # Homocedasticidad
    p_levene = levene(g1, g2)[1]
    homoced = p_levene > 0.05

    # Test estadístico
    if normal and homoced:
        _, p_val = ttest_ind(g1, g2)
        test = "t-test"
    else:
        _, p_val = mannwhitneyu(g1, g2)
        test = "Mann-Whitney"

    resultados.append([canal, test, p_val])


df_resultados = pd.DataFrame(resultados, columns=["Canal", "Prueba", "p-valor"])
df_resultados = df_resultados.sort_values("p-valor")

print("\nRESULTADOS:")
print(df_resultados)

significativos = df_resultados[df_resultados["p-valor"] < 0.05]

print("\nCANALES SIGNIFICATIVOS:")
print(significativos)


RESULTADOS:
   Canal        Prueba   p-valor
28  Af8.  Mann-Whitney  0.677585
37  F8..  Mann-Whitney  0.677585
19  Cp4.  Mann-Whitney  0.733730
13  C6..  Mann-Whitney  0.733730
36  F6..  Mann-Whitney  0.733730
..   ...           ...       ...
0   Fc5.        t-test  0.994273
58  Po4.        t-test  0.995059
43  T10.        t-test  0.997115
63  Iz..        t-test  0.999137
31  F3..        t-test  0.999572

[64 rows x 3 columns]

CANALES SIGNIFICATIVOS:
Empty DataFrame
Columns: [Canal, Prueba, p-valor]
Index: []


**RESULTADOS**

In [18]:
# 5. BOXPLOT
if len(significativos) > 0:

    canal_top = significativos.iloc[0]["Canal"]

    data_plot = pd.DataFrame({
        "RMS": pd.concat([df_T1[canal_top], df_T2[canal_top]]),
        "Grupo": ["Izquierda"]*len(df_T1) + ["Derecha"]*len(df_T2)
    })

    plt.figure()
    sns.boxplot(x="Grupo", y="RMS", data=data_plot)
    plt.title(f"Canal más significativo: {canal_top}")
    plt.show()

else:
    print("No se encontraron canales significativos")

No se encontraron canales significativos


**DISCUSIÓN**

Los resultados obtenidos indican que no se encontraron diferencias estadísticamente significativas (p < 0.05) entre las condiciones de imaginería motora de mano izquierda (T1) y mano derecha (T2) para ninguno de los canales EEG analizados. Este resultado se evidencia en los valores elevados de p-valor obtenidos en todas las pruebas estadísticas, así como en la ausencia de canales significativos en el análisis final.

Desde el punto de vista fisiológico, este resultado no coincide completamente con la evidencia reportada en la literatura, donde se espera observar una desincronización de la actividad neuronal en las cortezas motoras contralaterales, especialmente en canales como C3 y C4. Esta desincronización, conocida como ERD (Event-Related Desynchronization), se manifiesta como una disminución de la energía en bandas específicas de frecuencia durante la planificación del movimiento.

Una posible explicación de la ausencia de diferencias significativas radica en la naturaleza del descriptor utilizado. El valor RMS calcula la energía total de la señal en el dominio del tiempo, sin discriminar entre componentes espectrales. Sin embargo, la imaginería motora afecta principalmente bandas de frecuencia específicas, como la banda mu (8–12 Hz) y beta (13–30 Hz), por lo que el uso de una medida global como el RMS puede no ser suficientemente sensible para capturar estos cambios.

Adicionalmente, la variabilidad inter-sujeto representa otro factor relevante. Al analizar una población de múltiples sujetos, las diferencias individuales en la actividad cerebral pueden enmascarar los patrones comunes asociados a la tarea, reduciendo la capacidad estadística para detectar efectos significativos.

En conjunto, estos factores sugieren que, aunque la metodología implementada cumple con los requerimientos del análisis, el uso del RMS como única característica puede no ser adecuado para la discriminación de tareas de imaginería motora en señales EEG. Se podría emplear características en el dominio de la frecuencia, como la densidad espectral de potencia (PSD), que permiten analizar de manera más precisa los cambios en las bandas relevantes para este tipo de tareas.

**CONCLUSIONES**

Gracias a la práctica realizada se puede concluir que:

 1- El análisis de las señales EEG mediante el cálculo de la amplitud efectiva (RMS) permitió caracterizar la energía promedio de cada canal para las tareas de imaginería motora de mano izquierda (T1) y mano derecha (T2) en una población de 10 sujetos.

 2- el uso del RMS como única característica en el dominio del tiempo no es suficientemente sensible para detectar los cambios neurofisiológicos asociados a la imaginería motora, los cuales se manifiestan principalmente en bandas específicas de frecuencia.

 3- A pesar de no identificar canales diferenciales, la metodología implementada cumple con los requerimientos establecidos y proporciona una base sólida para el análisis de señales EEG en aplicaciones de interfaces cerebro-computadora (BCI).